# Index RAG sur Colab (embeddings bge-m3)

**Kabyle LLM Benchmark, phase 2, approche 2 (RAG).** Ce notebook calcule les *embeddings* (des vecteurs de nombres qui représentent le sens d'une phrase) des paires français–kabyle **décontaminées**. Il sert quand le PC n'a pas assez de mémoire pour le faire lui-même.

Le calcul utilise **exactement le même modèle que le labo** : `bge-m3` servi par Ollama. Les vecteurs calculés ici sont donc compatibles avec ceux que le labo calcule pour chaque question.

## Règle anti-contamination

Le seul fichier à téléverser est `rag_texts.jsonl`, produit **sur le PC** par :

```bash
docker compose run --rm api python -m pipelines.build_rag_index --limit 20000 --export-colab
```

Ces phrases viennent de corpus publics. Elles ont été filtrées sur le PC contre **tout** le jeu de test, partie privée comprise. **Aucun test ne part sur Colab** : ne jamais téléverser un fichier de `data/test/`.

## Étapes

1. *Exécution → Modifier le type d'exécution → T4 GPU*. Le CPU marche aussi, mais il est beaucoup plus lent.
2. Exécuter les cellules dans l'ordre et téléverser `rag_texts.jsonl` quand c'est demandé.
3. Télécharger `embeddings.npy` et le placer dans `data/train/processed/rag_colab/` sur le PC.
4. Sur le PC :

```bash
docker compose run --rm api python -m pipelines.build_rag_index --import-colab data/train/processed/rag_colab/embeddings.npy
```

Durée estimée (non mesurée) : environ 10 à 20 minutes pour 20 000 phrases sur T4.


In [ ]:
# ── Configuration ──────────────────────────────────────────
EMBED_MODEL = "bge-m3"  # identique au labo (meta.json de l'index)
BATCH = 64

In [ ]:
%%capture
# ── Installer et démarrer Ollama, puis télécharger bge-m3 ──
!sudo apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull {EMBED_MODEL}

In [ ]:
# ── Vérifier qu'Ollama répond et que le modèle est là ──────
import requests
tags = requests.get("http://127.0.0.1:11434/api/tags", timeout=10).json()
names = [m["name"] for m in tags.get("models", [])]
print(names)
assert any(n.split(":")[0] == EMBED_MODEL for n in names), "bge-m3 absent : relancer la cellule précédente"

In [ ]:
# ── Téléverser rag_texts.jsonl (produit sur le PC) ─────────
import json, os
try:
    from google.colab import files
    if not os.path.exists("rag_texts.jsonl"):
        files.upload()  # choisir data/train/processed/rag_colab/rag_texts.jsonl
except ImportError:
    pass
with open("rag_texts.jsonl", encoding="utf-8") as f:
    texts = [json.loads(line)["text"] for line in f if line.strip()]
print(len(texts), "textes à vectoriser")

In [ ]:
# ── Calcul des embeddings (même découpage que sur le PC) ───
import numpy as np
vectors = []
for start in range(0, len(texts), BATCH):
    r = requests.post("http://127.0.0.1:11434/api/embed",
                      json={"model": EMBED_MODEL, "input": texts[start:start + BATCH]}, timeout=600)
    r.raise_for_status()
    vectors.append(np.asarray(r.json()["embeddings"], dtype=np.float32))
    if (start // BATCH) % 20 == 0:
        print(f"{min(start + BATCH, len(texts))}/{len(texts)}")
matrix = np.vstack(vectors)
assert matrix.shape[0] == len(texts)
np.save("embeddings.npy", matrix)
print("embeddings.npy :", matrix.shape)

In [ ]:
# ── Télécharger le résultat ────────────────────────────────
try:
    from google.colab import files
    files.download("embeddings.npy")
except ImportError:
    print("Récupérer embeddings.npy dans le dossier courant")